In [ ]:
#| hide
from nbdev import show_doc

# Pairwise Relation Between Multiple Embeddings of Time Series

::: {.callout-note title="Pairwise relation methods (DTW & Distance Correlation)"}
focus on pairwise distances between samples or time series. They differ in what the distance represents and how it is used:

- *DTW*‑based distances quantify alignment cost under temporal warping. They operate directly on raw sequences and produce a pairwise distance matrix suitable for clustering, manifold learning (*MDS*, *t‑SNE*, *UMAP*), or *similarity transforms*.
- *Distance Correlation* operates on Euclidean distance matrices and measures statistical dependence between two multivariate samples. Unlike DTW, it is not an alignment metric but a dependence statistic: it detects any nonlinear relationship and is zero iff independent.

Both methods rely on pairwise distances, but they answer different questions:

- DTW asks “How similar are the shapes under alignment?”
- Distance Correlation asks “Are these variables dependent in any way?”
:::

In [ ]:
#| eval: false
from fhemb.utils.cutils import plot_pr_heatmap, depict_DTW

/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-05 00:57:37.498869: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more

## Pairwise Relation Heatmap

In [ ]:
#| eval: false

show_doc(plot_pr_heatmap, title_level=4, name="Depict pairwise relation as a heatmap")

---

#### Depict pairwise relation as a heatmap

```python

def plot_pr_heatmap(
    ts, # Array of shape (n_samples, n_features, n_timesteps).
    alignment_type:str='fastdtw', # Type of alignment method to use. Options:
'dcor' or DTW type ('fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div').
    gamma:int=1, # Smoothing parameter for Soft-DTW (only used for DTW-based alignment types):
- Small gamma → behaves like classic DTW (hard minimum)
- Large gamma → smoother, more diffused alignment
    dist:function=euclidean, # Alignment metric function used inside DTW. Only applies when alignment_type='fastdtw'.
Ignored for 'tslearn_dtw' (uses euclidean), 'soft_dtw'/'soft_dtw_div' (use squared euclidean),
and 'dcor'.
    p:int=3, # Parameter for Minkowski alignment metric (if used). Only applies to DTW-based alignment types.
    radius:int=1, # Radius parameter for the FastDTW algorithm. Only applies when alignment_type='fastdtw'.
    normalize:NoneType=None, # Normalization method for the pairwise relation.
Options: 'max', 'minmax', 'z-score', 'l1', 'l2', 'softmax', 'sym', 'log', 'none'.
    n_jobs:int=4, # Number of parallel jobs.
    labels:NoneType=None, # Labels for the time series. If None, defaults to TS0, TS1, ...
    title:str='Pairwise Relation Heatmap', # Title of the plot.
    color_continuous_scale:str='Viridis', # Colormap for the heatmap.
    base_size:int=20, # Base pixel size per time series (controls figure dimensions).
): # Displays an interactive Plotly heatmap.


```

*Visualize a pairwise relation matrix as an interactive Plotly heatmap with hover tooltips.*
The matrix is computed using either distance correlation (dcor) or a DTW-based
method depending on the selected alignment type.

::: {.callout collapse="true" title= "DTW Methods - Equations Summary (tslearn + fastdtw)"}
| `alignment_type` value | Underlying Function / Class                | Conceptual Metric        | Equation / Definition | Notes |
|------------------|--------------------------------------------|---------------------------|------------------------|-------|
| `tslearn_dtw`    | `tslearn_dtw(ts1, ts2)`                    | Dynamic Time Warping (DTW) | $$\mathrm{DTW}(X,Y) = \min_{\pi} \sum_{(i,j)\in\pi} \|X_i - Y_j\|$$ | Exact DTW; dynamic programming; $O(n^2)$. |
| `fastdtw`        | `fastdtw(ts1, ts2)`                        | FastDTW (approx. DTW)      | *No closed form* (approximation of DTW) | Linear‑time, linear‑space approximation; very fast. |
| `soft_dtw`       | `tslearn_soft_dtw(ts1, ts2, gamma)`        | Soft‑DTW                   | $$\text{SoftDTW}_\gamma(X,Y) = -\gamma \log \sum_{\pi} e^{-\frac{1}{\gamma} \sum_{(i,j)\in\pi}\|X_i-Y_j\|}$$ | Differentiable relaxation of DTW; controlled by `gamma`. |
| `soft_dtw_div`   | `soft_dtw_divergence(ts1, ts2, gamma)`     | Soft‑DTW Divergence        | $$D(X,Y) = \mathrm{SoftDTW}(X,Y) - \tfrac{1}{2}(\mathrm{SoftDTW}(X,X) + \mathrm{SoftDTW}(Y,Y))$$ | Proper divergence; symmetric; $\geq 0$. |
| `dcorrelation` | `dcor.distance_correlation(x, y)` | Distance Correlation (biased V‑statistic) | $$\mathrm{dCor}(X,Y)=\frac{\mathrm{dCov}(X,Y)}{\sqrt{\mathrm{dCov}(X,X)\,\mathrm{dCov}(Y,Y)}}$$ with $$\mathrm{dCov}^2(X,Y)=\frac{1}{n^2}\sum_{i,j} A_{ij}B_{ij}$$ where \(A,B\) are doubly‑centered distance matrices. | Detects any dependence; 0 iff independent; supports multivariate inputs; exponent $\in (0,2]$; multiple computation methods; $O(n^2)$. |
:::

::: {.callout-note collapse="true" title="gamma value"}
The **gamma ($\gamma$)** parameter controls how quickly similarity decays with distance in the Gaussian/RBF kernel. It effectively sets the *locality scale* of the embedding: small $\gamma$ captures global structure, large $\gamma$ emphasizes local neighborhoods. Here, $\mathbf{d}$ denotes a characteristic distance scale, e.g. the **median** of all pairwise distances in the distance matrix.

| $\gamma$ value (relative to distance scale) | Kernel Width | Effect on Similarity Matrix | Effect on 3D Embedding | When to Use |
|--------------------------------------|--------------|------------------------------|-------------------------|--------------|
| **Very small** ($\gamma ≈ 0.001 / d^2$)      | Very wide    | Most distances map to high similarity (matrix almost uniform) | Embedding collapses; points cluster together | When distances are extremely noisy; rarely useful |
| **Small** ($\gamma \approx 0.01 / d^2$)            | Wide         | Slow decay; far points still moderately similar | Smooth, global structure; weak cluster separation | When you want global geometry preserved |
| **Medium** ($\gamma \approx 0.1 / d^2$)            | Balanced     | Reasonable decay; neighbors clearly more similar | Balanced embedding: clusters visible, global shape preserved | **Good default** when scale is unknown |
| **Large** ($\gamma\approx 1 / d^2$)               | Narrow       | Only close neighbors have high similarity | Strong cluster separation; global shape distorted | When clusters matter more than global layout |
| **Very large** ($\gamma\geq 10 / d^2$)         | Very narrow  | Almost all similarities ≈ 0 except identical points | Embedding becomes noisy; overfitting to local noise | Only for extremely tight local structure |
:::

::: {.callout-caution title="gamma value"}
Applies to `soft_dtw` only.
::: 

::: {.callout collapse="true" title= "DTW Alignment Metrics - Summary Table"}
| `dist` value        | Function                | Equation | Description |
|---------------|--------------------------|----------|-------------|
| **Euclidean** | `euclidean(u, v)`        | $\sqrt{\sum_i (u_i - v_i)^2}$ | Standard L2 distance. |
| **Manhattan** | `cityblock(u, v)`        | $\sum_i \|u_i - v_i\|$ | L1 distance; sum of absolute differences. |
| **Cosine**    | `cosine(u, v)`           | $1 - \frac{u \cdot v}{\|u\|\|v\|}$ | Distance based on cosine similarity. |
| **Mahalanobis** | `mahalanobis(u, v, VI)` | $\sqrt{(u-v)^\top V^{-1}(u-v)}$ | Distance scaled by covariance structure. |
| **Squared Euclidean** | `sqeuclidean(u, v)` | $\sum_i (u_i - v_i)^2$ | L2 distance without the square root. |
| **Chebyshev** | `chebyshev(u, v)`        | $\max_i \|u_i - v_i\|$ | Maximum coordinate difference. |
| **Minkowski** | `minkowski(u, v, p)`     | $\left(\sum_i \|u_i - v_i\|^p\right)^{1/p}$ | Generalized Lp distance. |
| **Bray–Curtis** | `braycurtis(u, v)`      | $\frac{\sum_i \|u_i - v_i\|}{\sum_i \|u_i + v_i\|}$ | Normalized L1 emphasizing relative differences. |
| **Canberra**  | `canberra(u, v)`         | $\sum_i \frac{\|u_i - v_i\|}{\|u_i\| + \|v_i\|}$ | Weighted L1 emphasizing small values. |
| **Correlation** | `correlation(u, v)`     | $1 - \text{corr}(u, v)$ | Distance based on linear correlation. |
| **Jaccard**   | `jaccard(u, v)`          |  $1 - \frac{\|u \cap v\|}{\|u \cup v\|}$ | For binary/boolean vectors. |
| **Hamming**   | `hamming(u, v)`          | $\frac{1}{n}\sum_i [u_i \ne v_i]$ | Fraction of differing coordinates. |
:::

::: {.callout-caution title="dist value"}
Applies to `fastdtw` only.
:::

::: {.callout-note collapse="true" title="FastDTW: radius value"}
The `radius` parameter controls how wide the refinement window is around the projected DTW path in FastDTW.  
It must be a **non‑negative integer** ($0, 1, 2, \ldots $). Larger values improve accuracy at the cost of speed.

| `radius` value | Interpretation | Effect |
|--------------|----------------|--------|
| `0`          | Follow projected path exactly; no refinement | Fastest, least accurate |
| `1`          | Slight refinement around the path | Still fast; modest accuracy gain |
| `2`          | Balanced refinement window | Good trade‑off between speed and accuracy |
| `≥ 5`        | Wide refinement window | Much slower; very close to exact DTW |
:::

::: {.callout-caution title="radius value"}
Applies to `fastdtw` only.
:::

::: {.callout collapse="true" title= "Matrix Normalization Methods — API Summary"}
| `normalize` value | Meaning / Transformation | Equation | Scope | Strengths | Notes |
|-------------|--------------------------|----------|--------|-----------|-------|
| `max` | Global max normalization | $M \leftarrow M / \max(\|M\|)$ | Whole matrix | Simple global scaling; preserves sign | Sensitive to outliers; collapses if max is 0 |
| `minmax` | Min–max scaling to $[0,1]$ | $M \leftarrow \dfrac{M - \min(M)}{\max(M) - \min(M)}$ | Whole matrix | Uniform scaling; preserves relative ordering | If all values equal → returns zeros |
| `z-score` | Standardization (zero mean, unit variance) | $M \leftarrow \dfrac{M - \mu}{\sigma}$ | Whole matrix | Centers + scales; good for Gaussian‑like data | If $\sigma = 0$ → returns zeros |
| `l1` | Row‑wise L1 normalization | $M_{i,:} \leftarrow \dfrac{M_{i,:}}{\sum_j \|M_{i,j}\|}$ | Row‑wise | Produces rows summing to 1; good for sparse data | Zero rows remain zero (safe handling) |
| `l2` | Row‑wise L2 normalization | $M_{i,:} \leftarrow \dfrac{M_{i,:}}{\sqrt{\sum_j M_{i,j}^2}}$ | Row‑wise | Normalizes direction; common in embeddings | Zero rows remain zero (safe handling) |
| `softmax` | Row‑wise softmax (probabilities) | $M_{i,j} \leftarrow \dfrac{e^{M_{i,j}}}{\sum_k e^{M_{i,k}}}$ | Row‑wise | Converts rows into probability distributions | Uses max‑shift for numerical stability |
| `sym` | Symmetric graph normalization | $M \leftarrow D^{-1/2} \, M \, D^{-1/2}$ where $D_{ii} = \sum_j M_{ij}$ | Whole matrix | Standard in spectral graph theory; stabilizes adjacency matrices | Adds $1e{-8}$ to avoid division by zero |
| `log` | Logarithmic transform | $M \leftarrow \log(1 + M)$ | Element‑wise | Compresses large values; reduces skew | Requires non‑negative input for meaningful interpretation |
| `None` | No normalization | — | — | Leaves matrix unchanged | Useful for debugging or raw comparisons |
:::

## DTW Alignment

In [ ]:
#| eval: false
show_doc(depict_DTW, name="DTW alignment between two time series",  title_level=4)

---

#### DTW alignment between two time series

```python

def depict_DTW(
    x1:ndarray, # Time series where the first dimension is time and the second dimension are the features.
    x2:ndarray, # Time series where the first dimension is time and the second dimension are the features.
    dist:Callable=euclidean, # Alignment metric function to use for DTW.
    radius:int=1, # Radius parameter for the FastDTW algorithm.
):


```

*Use `fastdtw` to visualize the DTW alignment between two time series.*